# Jackson 2020: cell tables, masks and contact graph

This notebook builds the cell-level data that the Kaggle dataset `yuvalsamoilov/jackson2020` lacks; that dataset holds only images.
It is the Jackson counterpart of `danenberg-cells-prep`, so that `ProfileVsPhysical` and `KnnEnrichmentIMC` can be re-run on a second cohort.

**Source:** Zenodo record 4607374 (Jackson et al. 2020, *Nature*). The 37 GB and 11 GB zips are read with HTTP range requests, so only the needed members are downloaded.
- `SingleCell_and_Metadata.zip`: `BaselTMA/SC_dat.csv`, `ZurichTMA/SC_dat.csv` (long format: mean counts per cell and channel) and the patient metadata;
- `singlecell_locations.zip`: cell centroids;
- `singlecell_cluster_labels.zip`: the published metaclusters (Basel, and Zurich matched to Basel) and their annotation;
- `OMEandSingleCellMasks.zip` → `Basel_Zuri_masks.zip`: the published cell masks.

**Outputs** (`/kaggle/working`; later kernels attach them via `kernel_sources`):
- `cells.parquet`: one row per labelled cell. Channel columns use the Kaggle panel target names (35 channels, Kaggle image order) and hold the published mean counts.
  Other columns: image and patient ids, cohort, x/y (µm = px), area, metacluster phenotype, broad class and `is_tumour` (image level).
- `cell_neighbours.parquet`: contact graph (`ImageNumber`, `from`, `to`, both directions), computed from the masks because Jackson published none.
  Two cells are in contact if their masks come within `r` px. `r` is chosen so the mean degree matches the Danenberg published graph (5.87).
- `masks/<image_name>.tiff`: the published masks, relabelled to `object_number` (= Jackson `CellId`).
- `clinical.parquet`: patient metadata of both cohorts, one row per image.
- `prep_report.json`: every check listed in the last cell.

**Tumour images:** Basel `diseasestatus == "tumor"`; Zurich `location` in {CENTER, PERIPHERY}. Normal-breast Basel cores, Zurich metastases and Zurich cores with no location get `is_tumour = 0`.

**Channel Gd156** is listed as "Rabbit IgG H L" (the metal-tagged secondary). Its channel name is "…Gd156Di Estroge", i.e. it reads out the estrogen receptor, so it is renamed `estrogen_receptor_alpha`.

In [ ]:
# The Kaggle images are ZSTD-compressed TIFFs. This imagecodecs build works with the image's numpy 2.0.
!pip install -q "imagecodecs==2024.9.22"

In [ ]:
import os, io, re, json, glob, time, zipfile, shutil
import numpy as np, pandas as pd, requests, tifffile
from scipy.spatial import cKDTree

ZEN = "https://zenodo.org/records/4607374/files/{}?download=1"
OUT = "/kaggle/working"
os.makedirs(f"{OUT}/masks", exist_ok=True)
SEED = 20260927
DANENBERG_MEAN_DEGREE = 5.87   # Danenberg CellNeighbours.csv: 6,530,904 directed rows / 1,112,296 cells with a contact
REPORT = {}

# Kaggle image channel order (jackson2020 panel.parquet): target, metal, Jackson channel name
PANEL = [
    ("histone_h3", "In113", "1261726In113Di Histone"), ("histone_h3_trimethylate", "La139", "473968La139Di Histone"),
    ("cytokeratin_5", "Pr141", "651779Pr141Di Cytoker"), ("fibronectin", "Nd142", "3281668Nd142Di Fibrone"),
    ("cytokeratin_19", "Nd143", "3111576Nd143Di Cytoker"), ("cytokeratin_8_18", "Nd144", "971099Nd144Di Cytoker"),
    ("twist", "Nd145", "Nd145Di Twist"), ("cd68", "Nd146", "77877Nd146Di CD68"),
    ("keratin_14_krt14", "Sm147", "346876Sm147Di Keratin"), ("sma", "Nd148", "174864Nd148Di SMA"),
    ("vimentin", "Sm149", "1921755Sm149Di Vimenti"), ("c_myc", "Nd150", "322787Nd150Di cMyc"),
    ("c_erb_b_2_her2", "Eu151", "201487Eu151Di cerbB"), ("cd3", "Sm152", "8001752Sm152Di CD3epsi"),
    ("histone_h3_phospho", "Eu153", "phospho Histone"), ("slug", "Gd155", "3521227Gd155Di Slug"),
    ("estrogen_receptor_alpha", "Gd156", "112475Gd156Di Estroge"), ("progesterone_receptor_a_b", "Gd158", "312878Gd158Di Progest"),
    ("p53", "Tb159", "207736Tb159Di p53"), ("cd44", "Gd160", "6967Gd160Di CD44"),
    ("cd45", "Dy162", "71790Dy162Di CD45"), ("gata3", "Dy163", "117792Dy163Di GATA3"),
    ("cd20", "Dy164", "361077Dy164Di CD20"), ("carbonic_anhydrase_ix", "Er166", "92964Er166Di Carboni"),
    ("e_cadherin_p_cadherin", "Er167", "1031747Er167Di ECadhe"), ("ki_67", "Er168", "1441101Er168Di Ki67"),
    ("egfr", "Tm169", "1021522Tm169Di EGFR"), ("s6", "Er170", "phospho S6"),
    ("v_wf", "Yb172", "378871Yb172Di vWF"), ("m_tor", "Yb173", "phospho mTOR"),
    ("cytokeratin_7", "Yb174", "98922Yb174Di Cytoker"), ("pan_cytokeratin", "Lu175", "234832Lu175Di panCyto"),
    ("cleaved_parp", "Yb176", "198883Yb176Di cleaved"), ("dna1", "Ir191", "10331253Ir191Di Iridium"),
    ("dna2", "Ir193", "10331254Ir193Di Iridium"),
]
TARGETS = [p[0] for p in PANEL]

def find_input(pattern):
    """Kaggle has used several mount layouts; search all of /kaggle/input."""
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

img_files = find_input("*.tiff")
img_by_name = {os.path.splitext(os.path.basename(p))[0]: p for p in img_files}
print(f"{len(img_files)} Kaggle images")

class HTTPRangeFile(io.RawIOBase):
    """Seekable read-only file over HTTP range requests (with retries)."""
    def __init__(self, url):
        self.s = requests.Session()
        r = self.s.head(url, allow_redirects=True, timeout=60)
        self.url, self.size, self.pos = r.url, int(r.headers["Content-Length"]), 0
    def seekable(self): return True
    def readable(self): return True
    def tell(self): return self.pos
    def seek(self, off, whence=0):
        self.pos = {0: off, 1: self.pos + off, 2: self.size + off}[whence]
        return self.pos
    def readinto(self, b):
        n = min(len(b), self.size - self.pos)
        if n <= 0:
            return 0
        for attempt in range(8):
            try:
                r = self.s.get(self.url, timeout=300,
                               headers={"Range": f"bytes={self.pos}-{self.pos + n - 1}"})
                r.raise_for_status()
                data = r.content
                assert len(data) == n, f"short read {len(data)} != {n}"
                break
            except Exception as e:
                print(f"  range retry {attempt}: {e}")
                time.sleep(2 ** attempt)
        else:
            raise IOError("range request failed")
        b[:n] = data
        self.pos += n
        return n

def remote_zip(name, buf=64 << 20):
    """Large buffers keep the number of range requests small (Zenodo throttles many small ones)."""
    return zipfile.ZipFile(io.BufferedReader(HTTPRangeFile(ZEN.format(name)), buffer_size=buf))

TMP = next(d for d in ["/kaggle/tmp", "/tmp"] if os.path.isdir(d))
def extract(zf, member, dst):
    """Copy one (decompressed) zip member to local disk."""
    t0 = time.time()
    with zf.open(member) as src, open(dst, "wb") as f:
        shutil.copyfileobj(src, f, 16 << 20)
    print(f"  {member}: {os.path.getsize(dst) / 1e9:.2f} GB in {time.time() - t0:.0f}s")
    return dst

## Download the needed members

In [ ]:
t0 = time.time()
print(f"free space in {TMP}: {shutil.disk_usage(TMP).free / 1e9:.1f} GB")
scz = remote_zip("SingleCell_and_Metadata.zip")
P = "Data_publication"
meta_b = pd.read_csv(scz.open(f"{P}/BaselTMA/Basel_PatientMetadata.csv"))
meta_z = pd.read_csv(scz.open(f"{P}/ZurichTMA/Zuri_PatientMetadata.csv"))
sc_paths = {c: extract(scz, f"{P}/{d}/SC_dat.csv", f"{TMP}/sc_{c}.csv")
            for c, d in (("basel", "BaselTMA"), ("zurich", "ZurichTMA"))}
loc_z = remote_zip("singlecell_locations.zip", 8 << 20)
locs = {c: pd.read_csv(loc_z.open(f"{c.capitalize()}_SC_locations.csv")) for c in ("basel", "zurich")}
lab_z = remote_zip("singlecell_cluster_labels.zip", 8 << 20)
mc = pd.concat([pd.read_csv(lab_z.open("Cluster_labels/Basel_metaclusters.csv")),
                pd.read_csv(lab_z.open("Cluster_labels/Zurich_matched_metaclusters.csv"))])
ann = pd.read_csv(lab_z.open("Cluster_labels/Metacluster_annotations.csv"), sep=";")
ann.columns = [c.strip() for c in ann.columns]
ome = remote_zip("OMEandSingleCellMasks.zip", 8 << 20)
mask_zip = zipfile.ZipFile(io.BytesIO(ome.read("OMEnMasks/Basel_Zuri_masks.zip")))
print(f"downloads: {(time.time() - t0) / 60:.1f} min")
print(ann.to_string())

## Images, patients, tumour flag

In [ ]:
meta_b["cohort"], meta_z["cohort"] = "basel", "zurich"
meta = pd.concat([meta_b, meta_z], ignore_index=True)
meta["patient_id"] = np.where(meta.cohort == "basel", "B", "Z") + meta.PID.astype(str)
meta["is_tumour"] = (((meta.cohort == "basel") & (meta.diseasestatus == "tumor")) |
                     ((meta.cohort == "zurich") & meta.location.isin(["CENTER", "PERIPHERY"]))).astype(np.int8)

def kaggle_name(core):
    """BaselTMA_SP41_257_X3Y1 -> 41_257_x3y1; ZTMA208_slide_6_Ay4x3 -> 6_ay4x3 (the Kaggle file stems)."""
    return re.sub(r"^(BaselTMA_SP|ZTMA208_slide_)", "", core).lower()

meta["image_name"] = meta.core.map(kaggle_name)
assert meta.image_name.is_unique and meta.core.is_unique
meta = meta.sort_values("image_name").reset_index(drop=True)
meta["image_number"] = np.arange(1, len(meta) + 1, dtype=np.int32)
meta.astype(str).to_parquet(f"{OUT}/clinical.parquet", index=False)
REPORT["n_images_meta"] = int(len(meta))
REPORT["images_by_cohort_tumour"] = meta.groupby(["cohort", "is_tumour"]).size().rename("n").reset_index().to_dict("records")
REPORT["zurich_location"] = meta_z.location.value_counts(dropna=False).to_dict()
REPORT["meta_images_not_on_kaggle"] = sorted(set(meta.image_name) - set(img_by_name))
REPORT["kaggle_images_not_in_meta"] = sorted(set(img_by_name) - set(meta.image_name))
print(REPORT["images_by_cohort_tumour"])
print("not on kaggle:", len(REPORT["meta_images_not_on_kaggle"]), REPORT["meta_images_not_on_kaggle"][:10])
print("kaggle not in meta:", len(REPORT["kaggle_images_not_in_meta"]), REPORT["kaggle_images_not_in_meta"][:10])

## Cells: locations + metaclusters
Cell id = `<core>_<CellId>`; the location files carry it as `id` (built from `ObjectNumber_renamed`).
Cells without a published metacluster are dropped (counted in the report).

In [ ]:
loc = pd.concat(locs.values(), ignore_index=True)
REPORT["loc_objectnumber_renamed_equal_frac"] = float((loc.ObjectNumber == loc.ObjectNumber_renamed).mean())
loc = loc[loc.core.isin(meta.core)]
assert loc.id.is_unique
mc = mc.drop_duplicates("id")
ann["Metacluster"] = ann.Metacluster.astype(int)
ann["cell_type"] = ann["Cell type"].str.strip().replace({"Macrohage": "Macrophage"})
ann["phen"] = ann.Metacluster.map("MC{:02d} ".format) + ann.cell_type
ann["broad"] = ann.Class.str.strip().map({"Tumor": "epithelial", "Immune": "immune", "Stroma": "stromal", "Vessel": "stromal"})
assert ann.broad.notna().all()
cells = loc.merge(mc, on="id", how="left")
REPORT["n_cells_located"] = int(len(cells))
REPORT["frac_cells_without_metacluster"] = float(cells.cluster.isna().mean())
REPORT["frac_cells_without_metacluster_tumour_images"] = float(
    cells[cells.core.isin(meta.core[meta.is_tumour == 1])].cluster.isna().mean())
cells = cells[cells.cluster.notna()].copy()
cells["cluster"] = cells.cluster.astype(int)
m = meta.set_index("core")
a = ann.set_index("Metacluster")
cells = pd.DataFrame({
    "id": cells.id.values,
    "core": cells.core.values,
    "image_name": cells.core.map(m.image_name).values,
    "image_number": cells.core.map(m.image_number).values.astype(np.int32),
    "object_number": cells.ObjectNumber_renamed.values.astype(np.int32),
    "patient_id": cells.core.map(m.patient_id).values,
    "cohort": cells.core.map(m.cohort).values,
    "is_tumour": cells.core.map(m.is_tumour).values.astype(np.int8),
    "metacluster": cells.cluster.values.astype(np.int16),
    "cell_phenotype": cells.cluster.map(a.phen).values,
    "broad_class": cells.cluster.map(a.broad).values,
    "x": cells.Location_Center_X.values.astype(np.float32),
    "y": cells.Location_Center_Y.values.astype(np.float32),
})
assert cells.cell_phenotype.notna().all()
cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
print(f"{len(cells):,} labelled cells in {cells.image_name.nunique()} images;",
      f"{REPORT['frac_cells_without_metacluster']:.3f} of located cells have no metacluster")

## Marker values: long `SC_dat` → wide
Channels are matched to the Kaggle panel by metal tag (`<metal>Di`). The three channel names without a metal ("phospho …") are matched exactly.

In [ ]:
def panel_index(c):
    hit = [j for j, (t, metal, name) in enumerate(PANEL) if f"{metal}Di" in c or c == name]
    assert len(hit) <= 1, (c, hit)
    return hit[0] if hit else -1

idx = pd.Index(cells.id)
X = np.full((len(cells), len(PANEL)), np.nan, np.float32)
chan_seen = {}
t0 = time.time()
for coh, path in sc_paths.items():
    for ch in pd.read_csv(path, usecols=["id", "channel", "mc_counts"], chunksize=10_000_000,
                          dtype={"channel": "category", "mc_counts": np.float32}):
        cats = list(ch.channel.cat.categories)
        lut = np.array([panel_index(c) for c in cats] + [-1])
        codes = ch.channel.cat.codes.values.astype(np.int64)
        cj = lut[np.where(codes >= 0, codes, len(cats))]
        ri = idx.get_indexer(ch.id.values)
        ok = (ri >= 0) & (cj >= 0)
        X[ri[ok], cj[ok]] = ch.mc_counts.values[ok]
        for c, n in ch.channel.value_counts().items():
            chan_seen[str(c)] = chan_seen.get(str(c), 0) + int(n)
        print(f"  {coh}: chunk of {len(ch):,} rows, {time.time() - t0:.0f}s")
    os.remove(path)
REPORT["sc_channels"] = chan_seen
REPORT["channel_map"] = {t: [c for c in chan_seen if f"{metal}Di" in c or c == name] for t, metal, name in PANEL}
assert all(len(v) == 1 for v in REPORT["channel_map"].values()), REPORT["channel_map"]
REPORT["frac_missing_values"] = float(np.isnan(X).mean())
REPORT["frac_cells_with_missing"] = float(np.isnan(X).any(1).mean())
print("channels in SC_dat:", len(chan_seen), "| missing values:", REPORT["frac_missing_values"])
keep = ~np.isnan(X).any(1)
cells, X = cells[keep].reset_index(drop=True), X[keep]
for j, t in enumerate(TARGETS):
    cells[t] = X[:, j]
del X
REPORT["channel_quantiles"] = {t: np.round(np.quantile(cells[t], [0.01, 0.5, 0.99]), 3).tolist() for t in TARGETS}
print(pd.DataFrame(REPORT["channel_quantiles"], index=["q01", "q50", "q99"]).T.to_string())

## Masks: relabel to `object_number`, area, contact graph
- Each mask label is matched to its table cell by nearest centroid (< 2 px, one-to-one required). Unmatched labels become 0.
- The contact radius `r` (1–4 px, Chebyshev) is calibrated on 40 random tumour images: the `r` whose mean degree is closest to Danenberg's 5.87 is used for all images.

In [ ]:
mask_members = {re.sub(r"_full_ma(sk|ks)\.tiff$", "", os.path.basename(n)): n
                for n in mask_zip.namelist() if n.endswith(".tiff")}
meta["stem"] = meta.FileName_FullStack.str.replace(r"_full\.tiff$", "", regex=True)
REPORT["images_without_mask"] = sorted(meta.image_name[~meta.stem.isin(mask_members)])
print(len(mask_members), "masks;", len(REPORT["images_without_mask"]), "metadata images without a mask")

def contact_pairs(lab, r):
    """Unordered label pairs (a < b) whose pixels lie within r (Chebyshev) of each other."""
    H, W = lab.shape
    lab = lab.astype(np.int64)
    keys = []
    for dy in range(0, r + 1):
        for dx in range(-r, r + 1):
            if dy == 0 and dx <= 0:
                continue
            a = lab[0:H - dy, max(0, -dx):W - max(0, dx)]
            b = lab[dy:H, max(0, dx):W + min(0, dx)]
            m = (a > 0) & (b > 0) & (a != b)
            if m.any():
                keys.append(np.minimum(a[m], b[m]) * 65536 + np.maximum(a[m], b[m]))
    if not keys:
        return np.zeros((0, 2), np.int64)
    k = np.unique(np.concatenate(keys))
    return np.c_[k // 65536, k % 65536]

cells_by_img = {k: g for k, g in cells.groupby("image_name")}
mrow = meta.set_index("image_name")
relabelled, stats = {}, []
for name, g in cells_by_img.items():
    member = mask_members.get(mrow.stem[name])
    if member is None:
        continue
    msk = tifffile.imread(io.BytesIO(mask_zip.read(member))).astype(np.int64)
    u, inv = np.unique(msk, return_inverse=True)
    inv = inv.reshape(msk.shape)
    ys, xs = np.indices(msk.shape)
    cnt = np.bincount(inv.ravel())
    cx = np.bincount(inv.ravel(), xs.ravel()) / cnt
    cy = np.bincount(inv.ravel(), ys.ravel()) / cnt
    fg = np.where(u > 0)[0]
    dist, j = cKDTree(np.c_[cx[fg], cy[fg]]).query(g[["x", "y"]].values)
    ok = dist < 2.0
    lut = np.zeros(len(u), np.int64)
    lut[fg[j[ok]]] = g.object_number.values[ok]
    relabelled[name] = lut[inv].astype(np.uint16)
    stats.append(dict(image_name=name, h=msk.shape[0], w=msk.shape[1], n_table=len(g), n_mask=len(fg),
                      n_matched=int(ok.sum()), n_unique=int(len(np.unique(j[ok]))), dist_max=float(dist.max()),
                      label_eq_object=float(np.mean(u[fg[j[ok]]] == g.object_number.values[ok])) if ok.any() else np.nan))
ms = pd.DataFrame(stats)
ms.to_csv(f"{OUT}/mask_stats.csv", index=False)
print(ms.describe().round(3).to_string())

rng = np.random.default_rng(SEED)
tum_imgs = sorted(set(ms.image_name) & set(meta.image_name[meta.is_tumour == 1]))
calib = rng.choice(tum_imgs, min(40, len(tum_imgs)), replace=False)
deg = {}
for r in (1, 2, 3, 4):
    e = sum(2 * len(contact_pairs(relabelled[n], r)) for n in calib)
    c = sum(len(np.setdiff1d(np.unique(relabelled[n]), [0])) for n in calib)
    deg[r] = e / c
R_CONTACT = min(deg, key=lambda r: abs(deg[r] - DANENBERG_MEAN_DEGREE))
REPORT["contact_mean_degree_by_r"] = deg
REPORT["contact_r"] = R_CONTACT
print("mean degree by r:", {k: round(v, 2) for k, v in deg.items()}, "-> r =", R_CONTACT)

nb_rows = []
cells["area"] = np.float32(np.nan)
for name, relab in relabelled.items():
    pr = contact_pairs(relab, R_CONTACT)
    nb_rows.append(pd.DataFrame({"ImageNumber": int(mrow.image_number[name]),
                                 "from": np.r_[pr[:, 0], pr[:, 1]], "to": np.r_[pr[:, 1], pr[:, 0]]}))
    ar = np.bincount(relab.ravel().astype(np.int64))
    o = cells_by_img[name].object_number.values
    cells.loc[cells_by_img[name].index, "area"] = np.where(o < len(ar), ar[np.minimum(o, len(ar) - 1)], 0).astype(np.float32)
    tifffile.imwrite(f"{OUT}/masks/{name}.tiff", relab, compression="zlib")
nbr = pd.concat(nb_rows).astype(np.int32)
nbr.to_parquet(f"{OUT}/cell_neighbours.parquet", index=False)
REPORT["n_contacts"] = int(len(nbr))
REPORT["masks_written"] = int(len(relabelled))
REPORT["mask_cells_matched_frac"] = float(ms.n_matched.sum() / ms.n_table.sum())
REPORT["mask_one_to_one"] = bool((ms.n_unique == ms.n_matched).all())
REPORT["mask_dist_max"] = float(ms.dist_max.max())
REPORT["mask_label_eq_object_frac"] = float(np.nanmean(ms.label_eq_object))
REPORT["area_median"] = float(cells.area.median())
print(f"{len(nbr):,} directed contacts; matched {REPORT['mask_cells_matched_frac']:.4f}; "
      f"label == CellId {REPORT['mask_label_eq_object_frac']:.3f}; area median {REPORT['area_median']}")

## Alignment check against the Kaggle images
On 25 random images that have a Kaggle file, per-cell means from the Kaggle image with the relabelled mask must correlate with the published table values.
This checks both the mask relabelling and the channel mapping (Kaggle channel order = `PANEL`).

In [ ]:
def read_img(path):
    im = tifffile.imread(path)
    if im.ndim == 3 and im.shape[0] == len(PANEL) and im.shape[-1] != len(PANEL):
        im = np.moveaxis(im, 0, -1)
    return im  # H, W, C

chk_chan = ["histone_h3", "dna1", "pan_cytokeratin", "estrogen_receptor_alpha", "cd45", "ki_67", "sma"]
cand = sorted(set(relabelled) & set(img_by_name))
align, shape_ok = [], []
for name in rng.choice(cand, min(25, len(cand)), replace=False):
    im = read_img(img_by_name[name])
    relab = relabelled[name]
    shape_ok.append(im.shape[:2] == relab.shape)
    if im.shape[:2] != relab.shape or im.shape[2] != len(PANEL):
        print("shape mismatch", name, im.shape, relab.shape)
        continue
    g = cells_by_img[name]
    lab = relab.astype(np.int64).ravel()
    n_obj = int(max(g.object_number.max(), lab.max())) + 1
    npx = np.bincount(lab, minlength=n_obj)
    o = g.object_number.values
    for c in chk_chan:
        mean = np.bincount(lab, im[..., TARGETS.index(c)].ravel().astype(np.float64), n_obj) / np.maximum(npx, 1)
        align.append(dict(image_name=name, channel=c, r=float(np.corrcoef(mean[o], g[c].values)[0, 1])))
al = pd.DataFrame(align, columns=["image_name", "channel", "r"])
REPORT["kaggle_shape_match_frac"] = float(np.mean(shape_ok)) if shape_ok else None
REPORT["alignment_r_median_by_channel"] = al.groupby("channel").r.median().round(4).to_dict()
REPORT["alignment_r_min"] = float(al.r.min()) if len(al) else None
print(REPORT["alignment_r_median_by_channel"], "shape match:", REPORT["kaggle_shape_match_frac"])

## Save, report and hard checks

In [ ]:
cells = cells.drop(columns=["id"])
cells.to_parquet(f"{OUT}/cells.parquet", index=False)
tum = cells[cells.is_tumour == 1]
REPORT["n_cells"] = int(len(cells))
REPORT["n_images_table"] = int(cells.image_name.nunique())
REPORT["n_patients_table"] = int(cells.patient_id.nunique())
REPORT["n_tumour_images"] = int(tum.image_name.nunique())
REPORT["n_tumour_patients"] = int(tum.patient_id.nunique())
REPORT["tumour_cells_by_cohort"] = tum.cohort.value_counts().to_dict()
REPORT["phenotypes"] = cells.cell_phenotype.value_counts().to_dict()
REPORT["broad_classes"] = cells.broad_class.value_counts().to_dict()
REPORT["images_missing_on_kaggle"] = sorted(set(cells.image_name) - set(img_by_name))
checks = {
    "all_35_channels_mapped": len(REPORT["channel_map"]) == 35,
    "cells_>=500k": REPORT["n_cells"] >= 500_000,
    "tumour_images_>=400": REPORT["n_tumour_images"] >= 400,
    "mask_match_>=98%": REPORT["mask_cells_matched_frac"] >= 0.98,
    "mask_one_to_one": REPORT["mask_one_to_one"],
    "alignment_dna1_r>0.9": REPORT["alignment_r_median_by_channel"].get("dna1", 0) > 0.9,
}
REPORT["checks"] = checks
json.dump(REPORT, open(f"{OUT}/prep_report.json", "w"), indent=1, default=str)
print({k: REPORT[k] for k in ["n_cells", "n_images_table", "n_patients_table", "n_tumour_images", "n_tumour_patients"]})
print(json.dumps(checks, indent=1))
assert all(checks.values()), "prep checks failed - see prep_report.json"